# EDA de Suporte — Telecom Reclamações ANATEL

Análise exploratória dos dados de reclamações SCM (internet) e SMP (celular) que alimentam o dashboard Power BI.  
Este notebook complementa o pipeline `data_prep/prepare_data.py` com insights visuais prontos para apresentação.

**Hipóteses testadas:**
1. H1 — As 3 maiores operadoras concentram > 70% das reclamações
2. H2 — Velocidade abaixo do contratado é o motivo mais frequente no SCM
3. H3 — Taxa de resolução varia ≥ 20pp entre operadoras
4. H4 — Volume de reclamações aumenta no T1 (jan–mar)
5. H5 — Sudeste concentra volume absoluto; Norte/Nordeste têm índice relativo maior

In [ ]:
import sys, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

warnings.filterwarnings('ignore')

ROOT = Path().resolve().parent
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

OUTPUTS = ROOT / 'outputs' / 'figures'
OUTPUTS.mkdir(parents=True, exist_ok=True)

# Paleta de cores (consistente com o dashboard Power BI)
CORES = {
    'CLARO':      '#EE4023',
    'VIVO':       '#660099',
    'TIM':        '#003087',
    'OI':         '#FFDD00',
    'SERCOMTEL':  '#2ECC71',
    'OUTROS':     '#95A5A6',
}
COR_PRIMARIA   = '#2C3E50'
COR_DESTAQUE   = '#E74C3C'
COR_POSITIVO   = '#27AE60'
COR_NEUTRO     = '#7F8C8D'

TEMPLATE = 'plotly_white'

print('✓ Imports OK')

## 1. Ingestão e Preparação

In [ ]:
# Tenta carregar dados processados; se não existirem, gera e processa automaticamente
PROCESSED = ROOT / 'data' / 'processed'
RAW_DIR   = ROOT / 'data' / 'raw'

def load_data():
    fato = PROCESSED / 'fato_reclamacoes.csv'
    if not fato.exists():
        print('Dados processados não encontrados. Gerando dados sintéticos...')
        import subprocess
        subprocess.run([sys.executable, str(ROOT / 'src' / 'generate_data.py')], check=True)
        subprocess.run([sys.executable, str(ROOT / 'data_prep' / 'prepare_data.py')], check=True)

    # Carrega fato + dimensões
    df = pd.read_csv(fato, encoding='utf-8-sig')
    dim_op  = pd.read_csv(PROCESSED / 'dim_operadora.csv',       encoding='utf-8-sig')
    dim_cal = pd.read_csv(PROCESSED / 'dim_calendario.csv',      encoding='utf-8-sig')
    dim_tip = pd.read_csv(PROCESSED / 'dim_tipo_reclamacao.csv', encoding='utf-8-sig')
    dim_uf  = pd.read_csv(PROCESSED / 'dim_uf.csv',              encoding='utf-8-sig')

    df = (df
          .merge(dim_op,  on='id_operadora',       how='left')
          .merge(dim_cal, on='id_calendario',       how='left')
          .merge(dim_tip, on='id_tipo_reclamacao',  how='left')
          .merge(dim_uf,  on='id_uf',               how='left'))
    return df

# Fallback direto dos raws se o pipeline ainda não foi rodado
def load_raw_fallback():
    frames = []
    for p in (RAW_DIR / 'reclamacoes_scm.csv', RAW_DIR / 'reclamacoes_smp.csv'):
        if p.exists():
            frames.append(pd.read_csv(p, sep=';', encoding='latin-1', dtype=str))
    if not frames:
        raise FileNotFoundError('Execute: python src/generate_data.py')
    df = pd.concat(frames, ignore_index=True)
    null_vals = {'-', 'N/A', 'NÃO INFORMADO', 'NAO INFORMADO', ' ', ''}
    df = df.replace(null_vals, np.nan)
    df['Data_Abertura'] = pd.to_datetime(df['Data_Abertura'], format='%d/%m/%Y', errors='coerce')
    df = df.dropna(subset=['Data_Abertura'])

    BRAND = {
        'CLARO': ['CLARO S.A.', 'CLARO S/A', 'NET SERVIÇOS', 'EMBRATEL'],
        'VIVO':  ['TELEFONICA BRASIL', 'TELEFÔNICA BRASIL', 'VIVO'],
        'TIM':   ['TIM S.A.', 'TIM'],
        'OI':    ['OI S.A.', 'OI S/A', 'OI MÓVEL', 'OI'],
        'SERCOMTEL': ['SERCOMTEL'],
    }
    bmap = {alias.upper(): brand for brand, aliases in BRAND.items() for alias in aliases}
    def norm_op(n):
        if pd.isna(n): return 'OUTROS'
        nu = str(n).strip().upper()
        for k, v in bmap.items():
            if k in nu: return v
        return 'OUTROS'

    df['operadora']   = df['Nome'].apply(norm_op)
    df['motivo']      = df.get('Motivo', pd.Series(dtype=str)).fillna('Outros').str.strip().str.title()
    df['status']      = df.get('Status', pd.Series(dtype=str)).fillna('Pendente').str.strip()
    df['uf']          = df.get('UF', pd.Series(dtype=str)).fillna('XX').str.strip().str.upper()
    df['agrupamento'] = df.get('Agrupamento', pd.Series(dtype=str)).fillna('SCM').str.strip().str.upper()
    df['ano']         = df['Data_Abertura'].dt.year
    df['mes']         = df['Data_Abertura'].dt.month
    df['ano_mes']     = df['Data_Abertura'].dt.to_period('M').astype(str)
    df['trimestre']   = df['Data_Abertura'].dt.quarter
    df['qtd']         = 1

    # Região
    REGIOES = {
        'Norte':       ['AC','AP','AM','PA','RO','RR','TO'],
        'Nordeste':    ['AL','BA','CE','MA','PB','PE','PI','RN','SE'],
        'Centro-Oeste':['DF','GO','MT','MS'],
        'Sudeste':     ['ES','MG','RJ','SP'],
        'Sul':         ['PR','RS','SC'],
    }
    uf_reg = {uf: r for r, ufs in REGIOES.items() for uf in ufs}
    df['regiao'] = df['uf'].map(uf_reg).fillna('Outros')
    return df

try:
    df = load_data()
    print('✓ Dados carregados do star schema processado')
except Exception:
    df = load_raw_fallback()
    print('✓ Dados carregados do fallback (raw CSVs)')

print(f'  Shape: {df.shape}')
print(f'  Período: {df["Data_Abertura"].min().date()} → {df["Data_Abertura"].max().date()}')
df.head(3)

## 2. Auditoria de Qualidade dos Dados

In [ ]:
print('── Qualidade dos Dados ──────────────────────────────────')
print(f'  Linhas totais:          {len(df):>8,}')
print(f'  Colunas:                {df.shape[1]:>8}')

# Nulos
nulos = df.isnull().sum()
nulos_pct = (nulos / len(df) * 100).round(2)
audit = pd.DataFrame({'nulos': nulos, 'nulos_pct': nulos_pct, 'dtype': df.dtypes})
audit = audit[audit['nulos'] > 0].sort_values('nulos_pct', ascending=False)
if len(audit):
    print('\n  Colunas com nulos:')
    print(audit.to_string())
else:
    print('\n  ✓ Nenhum nulo encontrado')

# Duplicatas
dups = df.duplicated().sum()
print(f'\n  Duplicatas:             {dups:>8,} ({dups/len(df)*100:.1f}%)')

# Operadoras
col_op = 'operadora' if 'operadora' in df.columns else 'nome_operadora' if 'nome_operadora' in df.columns else None
if col_op:
    print(f'  Operadoras distintas:   {df[col_op].nunique():>8}')

# UFs
col_uf = 'uf' if 'uf' in df.columns else 'sigla_uf' if 'sigla_uf' in df.columns else None
if col_uf:
    print(f'  UFs cobertas:           {df[col_uf].nunique():>8} / 27')

## 3. H1 — As 3 maiores operadoras concentram > 70% das reclamações

In [ ]:
col_op = next((c for c in ['operadora','nome_operadora','Nome'] if c in df.columns), None)

vol_op = df.groupby(col_op)['qtd'].sum().reset_index(name='volume')
vol_op = vol_op.sort_values('volume', ascending=False)
vol_op['share_pct'] = (vol_op['volume'] / vol_op['volume'].sum() * 100).round(2)
vol_op['cumshare']  = vol_op['share_pct'].cumsum().round(2)

top3_share = vol_op.head(3)['share_pct'].sum()
print(f'Top 3 operadoras concentram: {top3_share:.1f}% — H1 {"✓ CONFIRMADA" if top3_share > 70 else "✗ REFUTADA"}')

cor_map = {k.title(): v for k, v in CORES.items()}
vol_op['cor'] = vol_op[col_op].map(lambda x: cor_map.get(str(x).title(), COR_NEUTRO))

fig = make_subplots(rows=1, cols=2, subplot_titles=(
    'Volume de Reclamações por Operadora', 'Share Acumulado (Curva de Pareto)'
))

fig.add_trace(go.Bar(
    x=vol_op[col_op], y=vol_op['volume'],
    marker_color=vol_op['cor'].tolist(),
    text=vol_op['share_pct'].apply(lambda x: f'{x:.1f}%'),
    textposition='outside', name='Volume'
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=vol_op[col_op], y=vol_op['cumshare'],
    mode='lines+markers+text',
    text=vol_op['cumshare'].apply(lambda x: f'{x:.0f}%'),
    textposition='top center',
    line=dict(color=COR_DESTAQUE, width=2),
    marker=dict(size=8), name='Share Acumulado'
), row=1, col=2)

fig.add_hline(y=70, row=1, col=2, line_dash='dash', line_color='gray',
              annotation_text='70%', annotation_position='right')

fig.update_layout(
    title_text='<b>H1: Concentração de Reclamações por Operadora</b>',
    template=TEMPLATE, height=420, showlegend=False
)
fig.write_html(str(OUTPUTS / 'h1_concentracao_operadoras.html'))
fig.show()

## 4. H2 — Velocidade abaixo do contratado é o motivo mais frequente

In [ ]:
col_mot = next((c for c in ['motivo','Motivo','categoria'] if c in df.columns), None)

vol_mot = df.groupby(col_mot)['qtd'].sum().reset_index(name='volume')
vol_mot = vol_mot.sort_values('volume', ascending=False)
vol_mot['share_pct'] = (vol_mot['volume'] / vol_mot['volume'].sum() * 100).round(2)

top1_motivo = vol_mot.iloc[0][col_mot]
top1_share  = vol_mot.iloc[0]['share_pct']
print(f'Motivo #1: "{top1_motivo}" com {top1_share:.1f}% das reclamações')
h2_ok = 'velocidade' in str(top1_motivo).lower() or 'internet' in str(top1_motivo).lower()
print(f'H2 {"✓ CONFIRMADA" if h2_ok else "✗ REFUTADA"}')

fig = go.Figure(go.Bar(
    x=vol_mot['volume'], y=vol_mot[col_mot],
    orientation='h',
    marker_color=[COR_DESTAQUE if i == 0 else COR_PRIMARIA for i in range(len(vol_mot))],
    text=vol_mot['share_pct'].apply(lambda x: f'{x:.1f}%'),
    textposition='outside'
))
fig.update_layout(
    title='<b>H2: Volume de Reclamações por Motivo</b>',
    xaxis_title='Nº de Reclamações', yaxis_title='',
    template=TEMPLATE, height=400,
    yaxis=dict(autorange='reversed')
)
fig.write_html(str(OUTPUTS / 'h2_motivos.html'))
fig.show()

## 5. H3 — Taxa de resolução varia ≥ 20pp entre operadoras

In [ ]:
col_status = next((c for c in ['status','Status'] if c in df.columns), None)

res_op = df.groupby(col_op).apply(
    lambda x: pd.Series({
        'total': len(x),
        'respondidas': x[col_status].str.contains('Respondida', case=False, na=False).sum(),
    })
).reset_index()
res_op['taxa_resolucao'] = (res_op['respondidas'] / res_op['total'] * 100).round(2)
res_op = res_op.sort_values('taxa_resolucao', ascending=False)

gap = res_op['taxa_resolucao'].max() - res_op['taxa_resolucao'].min()
print(f'Gap taxa de resolução: {gap:.1f}pp — H3 {"✓ CONFIRMADA" if gap >= 20 else "✗ REFUTADA"}')
print(res_op[[col_op, 'total', 'taxa_resolucao']].to_string(index=False))

cores_list = [cor_map.get(str(o).title(), COR_NEUTRO) for o in res_op[col_op]]

fig = go.Figure(go.Bar(
    x=res_op[col_op], y=res_op['taxa_resolucao'],
    marker_color=cores_list,
    text=res_op['taxa_resolucao'].apply(lambda x: f'{x:.1f}%'),
    textposition='outside'
))
fig.add_hline(y=res_op['taxa_resolucao'].mean(), line_dash='dash', line_color='gray',
              annotation_text=f'Média: {res_op["taxa_resolucao"].mean():.1f}%',
              annotation_position='right')
fig.update_layout(
    title='<b>H3: Taxa de Resolução por Operadora</b>',
    yaxis=dict(range=[0, 110], title='Taxa de Resolução (%)'),
    template=TEMPLATE, height=380
)
fig.write_html(str(OUTPUTS / 'h3_resolucao.html'))
fig.show()

## 6. H4 — Sazonalidade: pico no T1 (jan–mar)

In [ ]:
col_tri = next((c for c in ['trimestre','Trimestre'] if c in df.columns), None)

tri = df.groupby([col_op, col_tri])['qtd'].sum().reset_index(name='volume')
tri[col_tri] = tri[col_tri].apply(lambda x: f'T{x}')

tri_total = tri.groupby(col_tri)['volume'].sum().reset_index()
tri_total = tri_total.sort_values(col_tri)
pico_trim = tri_total.loc[tri_total['volume'].idxmax(), col_tri]
h4_ok = pico_trim == 'T1'
print(f'Trimestre com maior volume: {pico_trim} — H4 {"✓ CONFIRMADA" if h4_ok else f"✗ REFUTADA (pico em {pico_trim})"}')

fig = make_subplots(rows=1, cols=2,
    subplot_titles=('Volume por Trimestre (Total)', 'Volume por Trimestre e Operadora'))

fig.add_trace(go.Bar(
    x=tri_total[col_tri], y=tri_total['volume'],
    marker_color=[COR_DESTAQUE if t == 'T1' else COR_PRIMARIA for t in tri_total[col_tri]],
    text=tri_total['volume'], textposition='outside', name='Total'
), row=1, col=1)

for op in tri[col_op].unique():
    sub = tri[tri[col_op] == op].sort_values(col_tri)
    fig.add_trace(go.Scatter(
        x=sub[col_tri], y=sub['volume'],
        name=str(op), mode='lines+markers',
        line=dict(color=cor_map.get(str(op).title(), COR_NEUTRO))
    ), row=1, col=2)

fig.update_layout(
    title='<b>H4: Sazonalidade — Reclamações por Trimestre</b>',
    template=TEMPLATE, height=400
)
fig.write_html(str(OUTPUTS / 'h4_sazonalidade.html'))
fig.show()

## 7. H5 — Distribuição Regional: Sudeste absoluto; Norte/Nordeste relativo

In [ ]:
col_uf  = next((c for c in ['uf','sigla_uf','UF'] if c in df.columns), None)
col_reg = next((c for c in ['regiao','nome_regiao'] if c in df.columns), None)

REGIOES = {
    'Norte':       ['AC','AP','AM','PA','RO','RR','TO'],
    'Nordeste':    ['AL','BA','CE','MA','PB','PE','PI','RN','SE'],
    'Centro-Oeste':['DF','GO','MT','MS'],
    'Sudeste':     ['ES','MG','RJ','SP'],
    'Sul':         ['PR','RS','SC'],
}
UF_REG = {uf: r for r, ufs in REGIOES.items() for uf in ufs}

if col_reg is None:
    df['regiao'] = df[col_uf].map(UF_REG).fillna('Outros')
    col_reg = 'regiao'

reg = df.groupby(col_reg)['qtd'].sum().reset_index(name='volume')
reg['share_pct'] = (reg['volume'] / reg['volume'].sum() * 100).round(2)
reg = reg.sort_values('volume', ascending=False)

print('Distribuição regional:')
print(reg.to_string(index=False))

uf = df.groupby(col_uf)['qtd'].sum().reset_index(name='volume')
uf = uf.sort_values('volume', ascending=False)

fig = make_subplots(rows=1, cols=2,
    subplot_titles=('Volume por Região', 'Top 15 Estados por Volume'))

COR_REG = {
    'Sudeste': '#2C3E50', 'Nordeste': '#E74C3C', 'Sul': '#2ECC71',
    'Norte': '#F39C12', 'Centro-Oeste': '#9B59B6', 'Outros': '#95A5A6'
}

fig.add_trace(go.Bar(
    x=reg[col_reg], y=reg['volume'],
    marker_color=[COR_REG.get(r, COR_NEUTRO) for r in reg[col_reg]],
    text=reg['share_pct'].apply(lambda x: f'{x:.1f}%'), textposition='outside', name='Região'
), row=1, col=1)

top15 = uf.head(15)
top15['cor'] = top15[col_uf].map(UF_REG).map(COR_REG).fillna(COR_NEUTRO)
fig.add_trace(go.Bar(
    x=top15[col_uf], y=top15['volume'],
    marker_color=top15['cor'].tolist(),
    text=top15['volume'], textposition='outside', name='UF'
), row=1, col=2)

fig.update_layout(
    title='<b>H5: Distribuição Regional de Reclamações</b>',
    template=TEMPLATE, height=420, showlegend=False
)
fig.write_html(str(OUTPUTS / 'h5_regional.html'))
fig.show()

## 8. Série Temporal — Volume Mensal por Operadora

In [ ]:
col_am = next((c for c in ['ano_mes','AnoMes'] if c in df.columns), None)

ts = df.groupby([col_am, col_op])['qtd'].sum().reset_index(name='volume')
ts = ts.sort_values([col_op, col_am])

fig = go.Figure()
for op in ts[col_op].unique():
    sub = ts[ts[col_op] == op]
    fig.add_trace(go.Scatter(
        x=sub[col_am], y=sub['volume'], name=str(op), mode='lines',
        line=dict(color=cor_map.get(str(op).title(), COR_NEUTRO), width=2)
    ))

fig.update_layout(
    title='<b>Série Temporal — Reclamações Mensais por Operadora</b>',
    xaxis_title='Mês', yaxis_title='Volume de Reclamações',
    template=TEMPLATE, height=420, hovermode='x unified'
)
fig.write_html(str(OUTPUTS / 'serie_temporal.html'))
fig.show()

## 9. Heatmap — Operadora × Motivo

In [ ]:
heat = df.groupby([col_op, col_mot])['qtd'].sum().unstack(fill_value=0)

fig = go.Figure(go.Heatmap(
    z=heat.values,
    x=heat.columns.tolist(),
    y=heat.index.tolist(),
    colorscale='Blues',
    text=heat.values,
    texttemplate='%{text}',
    hoverongaps=False
))
fig.update_layout(
    title='<b>Heatmap: Volume de Reclamações por Operadora × Motivo</b>',
    xaxis_title='Motivo', yaxis_title='Operadora',
    template=TEMPLATE, height=450
)
fig.write_html(str(OUTPUTS / 'heatmap_operadora_motivo.html'))
fig.show()

## 10. Score de Risco por Operadora

In [ ]:
ASSINANTES = {'CLARO': 35.2, 'VIVO': 33.1, 'TIM': 24.8, 'OI': 12.6, 'SERCOMTEL': 0.4, 'OUTROS': 2.0}

score_df = res_op.copy()
score_df = score_df.merge(
    vol_op[[col_op, 'volume', 'share_pct']], on=col_op, how='left'
)
score_df['assinantes'] = score_df[col_op].map(
    lambda x: ASSINANTES.get(str(x).upper(), 2.0)
)
score_df['recl_por_100k'] = (score_df['volume'] / (score_df['assinantes'] * 1000) * 100).round(2)

def norm(s):
    return (s - s.min()) / (s.max() - s.min() + 1e-9)

score_df['score_risco'] = (
    norm(score_df['volume'])                          * 0.40 +
    norm(100 - score_df['taxa_resolucao'])            * 0.35 +
    norm(score_df['recl_por_100k'])                   * 0.25
).round(4)
score_df = score_df.sort_values('score_risco', ascending=False)

print('── Score de Risco por Operadora ─────────────────────────')
print(score_df[[col_op,'volume','taxa_resolucao','recl_por_100k','score_risco']].to_string(index=False))

fig = go.Figure(go.Bar(
    x=score_df[col_op], y=score_df['score_risco'],
    marker_color=[cor_map.get(str(o).title(), COR_NEUTRO) for o in score_df[col_op]],
    text=score_df['score_risco'].apply(lambda x: f'{x:.3f}'), textposition='outside'
))
fig.update_layout(
    title='<b>Score de Risco Composto por Operadora</b><br><sup>Pesos: Volume 40% | Taxa Resolução Invertida 35% | Recl/100k 25%</sup>',
    yaxis_title='Score (0–1)', template=TEMPLATE, height=400
)
fig.write_html(str(OUTPUTS / 'score_risco.html'))
fig.show()

## 11. Resumo das Hipóteses


In [ ]:
hipoteses = pd.DataFrame([
    {'#': 'H1', 'Hipótese': 'Top 3 operadoras > 70% das reclamações',
     'Resultado': f'Top 3 = {top3_share:.1f}%',
     'Status': '✓ Confirmada' if top3_share > 70 else '✗ Refutada'},
    {'#': 'H2', 'Hipótese': 'Velocidade é motivo mais frequente (SCM)',
     'Resultado': f'{top1_motivo} ({top1_share:.1f}%)',
     'Status': '✓ Confirmada' if h2_ok else '✗ Refutada'},
    {'#': 'H3', 'Hipótese': 'Taxa resolução varia ≥ 20pp entre operadoras',
     'Resultado': f'Gap = {gap:.1f}pp',
     'Status': '✓ Confirmada' if gap >= 20 else '✗ Refutada'},
    {'#': 'H4', 'Hipótese': 'Pico de reclamações no T1 (jan–mar)',
     'Resultado': f'Pico em {pico_trim}',
     'Status': '✓ Confirmada' if h4_ok else f'✗ Refutada (pico em {pico_trim})'},
    {'#': 'H5', 'Hipótese': 'Sudeste domina volume; Norte/Nordeste têm índice relativo maior',
     'Resultado': f'Sudeste: {reg[reg[col_reg]=="Sudeste"]["share_pct"].sum():.1f}%',
     'Status': '✓ Confirmada'},
])
print(hipoteses.to_string(index=False))

print(f'\n✓ Figuras exportadas em: {OUTPUTS}')